# Lab 2: Fine-Tune

### Workshop notebooks

| # | Notebook | What you'll do |
|---|---|---|
| 1 | `1-prepare-data.ipynb` | Format ContractNLI for SFT and register the datasets in SageMaker |
| 2 | `2-fine-tune-llm.ipynb` | Launch the serverless LoRA fine-tuning job on Nemotron 3 Nano 30B |
| 3 | `3-evaluation.ipynb` | Score the base, fine-tuned, and frontier models |
| 4 | `4-deployment.ipynb` | Deploy the fine-tuned model to a SageMaker real-time endpoint |

---

We now adapt Nemotron 3 Nano 30B to the contract review checklist using **LoRA** on SageMaker AI serverless customization. No cluster to provision, no container to build.

### Choosing a base model

The base model used across this lab is configured in [`config.py`](config.py). Want to
try a different model? Browse the available models on SageMaker JumpStart by running
the cell below. To switch, update `BASE_MODEL_ID` in `config.py`, all notebooks in
this lab pick up the change automatically.

> **Note:** Not every JumpStart model supports every customization technique. A
> "No recipes found" error means the model does not support SFT. Support for
> *serving* is a separate question, check that your chosen model can be served by
> the LMI/DJL container (notebook 4) before committing to it.


In [1]:
import boto3

from config import BASE_MODEL_ID

sm = boto3.client("sagemaker")
models, kwargs = [], {"HubName": "SageMakerPublicHub", "HubContentType": "Model",
                      "MaxResults": 100}
while True:
    r = sm.list_hub_contents(**kwargs)
    for item in r["HubContentSummaries"]:
        if "@capability:customization" in item.get("HubContentSearchKeywords", []):
            models.append(item["HubContentName"])
    if "NextToken" in r:
        kwargs["NextToken"] = r["NextToken"]
    else:
        break

print(f"base model for this lab: {BASE_MODEL_ID}\n")
print(f"customizable models available ({len(models)}):")
print("\n".join(sorted(models)))

base model for this lab: huggingface-reasoning-nvidia-nemotron-3-nano-30b-a3b-bf16

customizable models available (36):
deepseek-llm-r1-distill-llama-70b
deepseek-llm-r1-distill-llama-8b
deepseek-llm-r1-distill-qwen-1-5b
deepseek-llm-r1-distill-qwen-14b
deepseek-llm-r1-distill-qwen-32b
deepseek-llm-r1-distill-qwen-7b
huggingface-llm-nvidia-nemotron-3-super-120b-a12b-bf16
huggingface-llm-qwen2-5-14b-instruct
huggingface-llm-qwen2-5-32b-instruct
huggingface-llm-qwen2-5-72b-instruct
huggingface-llm-qwen2-5-7b-instruct
huggingface-reasoning-nvidia-nemotron-3-nano-30b-a3b-bf16
huggingface-reasoning-qwen3-06b
huggingface-reasoning-qwen3-1-7b
huggingface-reasoning-qwen3-14b
huggingface-reasoning-qwen3-32b
huggingface-reasoning-qwen3-4b
huggingface-reasoning-qwen3-8b
huggingface-vlm-gemma-4-26b-a4b-it
huggingface-vlm-gemma-4-31b-it
huggingface-vlm-gemma-4-e4b-it
huggingface-vlm-qwen3-5-27b
huggingface-vlm-qwen3-5-4b
huggingface-vlm-qwen3-5-9b
huggingface-vlm-qwen3-6-27b
meta-textgeneration-lla

In [ ]:
%load_ext autoreload
%autoreload 2

#### Setup and dependencies

In [3]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker role arn: arn:aws:iam::492681118881:role/service-role/AmazonSageMaker-ExecutionRole-20201215T102238
sagemaker bucket: sagemaker-us-east-1-492681118881
sagemaker session region: us-east-1


In [4]:
from sagemaker.ai_registry.dataset import DataSet
from config import BASE_MODEL_ID, DATASET_PREFIX

base_model_id = BASE_MODEL_ID
training_dataset = DataSet.get(name=f"{DATASET_PREFIX}-train")
val_dataset = DataSet.get(name=f"{DATASET_PREFIX}-val")

output_path = (f"s3://{bucket_name}/{default_prefix}/{base_model_id}-contractnli"
               if default_prefix else f"s3://{bucket_name}/{base_model_id}-contractnli")
print(f"training output: {output_path}")

[09/10/26 01:59:12] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=680523;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=181691;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#322\322]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=445637;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=944831;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=128758;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=894742;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#65\65]8;;\

training output: s3://sagemaker-us-east-1-492681118881/huggingface-reasoning-nvidia-nemotron-3-nano-30b-a3b-bf16-contractnli


### Create the Model Package Group

The trained model is registered here, and notebooks 3 and 4 look it up by this name.

In [5]:
import hashlib

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

# SageMaker caps Model Package Group names at 63 characters. Hash-truncate when
# base_model_id + suffix exceeds it, so notebooks 2, 3, 4 and 4a all derive the
# same name for any model id.
MAX_MPG_NAME_LENGTH = 63
suffix = "-contractnli-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

print(f"Model Package Group: {model_package_group_name}")


Model Package Group: huggingface-reasoning-nvidia-nemotro-b73b16-contractnli-sft-mpg


In [6]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

try:
    ModelPackageGroup.get(model_package_group_name=model_package_group_name)
    print(f"already exists: {model_package_group_name}")
except ClientError:
    ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="ContractNLI NDA checklist review, serverless SFT",
    )
    print(f"created: {model_package_group_name}")

[09/10/26 01:59:25] WARNING  No region provided. Using default region.                                 ]8;id=299121;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=831407;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

already exists: huggingface-reasoning-nvidia-nemotro-b73b16-contractnli-sft-mpg


### Configure the trainer

In [7]:
from sagemaker.train.common import TrainingType
from sagemaker.train.sft_trainer import SFTTrainer

MAX_JOB_NAME_LENGTH, TIMESTAMP_LENGTH = 63, 15
base_job_name = "contractnli-sft"[: MAX_JOB_NAME_LENGTH - TIMESTAMP_LENGTH].rstrip("-")

trainer = SFTTrainer(
    model=base_model_id,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group_name,
    training_dataset=training_dataset,
    validation_dataset=val_dataset,
    s3_output_path=output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
    base_job_name=base_job_name,
)

print(f"fine-tuning:    {base_model_id}")
print(f"training type:  {TrainingType.LORA.value}")
print(f"model package:  {model_package_group_name}")
print(f"training data:  {training_dataset.name} v{training_dataset.version}")
print(f"validation:     {val_dataset.name} v{val_dataset.version}\n")

print("default hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

fine-tuning:    huggingface-reasoning-nvidia-nemotron-3-nano-30b-a3b-bf16
training type:  LORA
model package:  huggingface-reasoning-nvidia-nemotro-b73b16-contractnli-sft-mpg
training data:  contractnli-nda-review-nothink-train v4.0.0
validation:     contractnli-nda-review-nothink-val v4.0.0

default hyperparameters:
  global_batch_size: 128
  learning_rate: 0.0001
  lora_alpha: 16
  lora_rank: 128
  lr_scheduler: cosine
  lr_warmup_steps_ratio: 0.0
  max_epochs: 2
  min_lr: 1e-06
  mlflow_run_id: 
  mlflow_tracking_uri: 
  model_name_or_path: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  name: example-name-5x0q0
  results_directory: 
  train_val_split_ratio: 1.0
  warmup_steps: 10
  weight_decay: 0.1


### Hyperparameters

The cell below sets six: the LoRA rank and alpha, the batch size, the epoch count, the
learning rate and the warmup ratio. The values are the ones the published run used.

Unlike the Qwen3 recipe, the Nemotron recipe exposes no `dataset_max_len`, so there is
no sequence cap to raise and no records are dropped for length. All 423 contracts train.

`global_batch_size` at 128 and `learning_rate` at 1e-4 already match the recipe defaults.


In [8]:
trainer.hyperparameters.global_batch_size = 128
trainer.hyperparameters.max_epochs = 10
trainer.hyperparameters.learning_rate = 0.0001
trainer.hyperparameters.lr_warmup_steps_ratio = 0.1
trainer.hyperparameters.lora_rank = 32
trainer.hyperparameters.lora_alpha = 64

records = 423                         # one per contract, as written by notebook 1

gbs = int(trainer.hyperparameters.global_batch_size)
epochs = int(trainer.hyperparameters.max_epochs)
ratio = float(trainer.hyperparameters.lr_warmup_steps_ratio)

# Batches are carried across epoch boundaries rather than each epoch ending on a short
# one, so the step total is the floor of records x epochs / batch size.
steps = records * epochs // gbs

print(f"{records} records train")
print(f"{records / gbs:.1f} steps/epoch x {epochs} epochs = {steps} optimizer steps")
print(f"warmup = {ratio:.0%} = {round(ratio * steps)} steps\n")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")


423 records train
3.3 steps/epoch x 10 epochs = 33 optimizer steps
warmup = 10% = 3 steps

  global_batch_size: 128
  learning_rate: 0.0001
  lora_alpha: 64
  lora_rank: 32
  lr_scheduler: cosine
  lr_warmup_steps_ratio: 0.1
  max_epochs: 10
  min_lr: 1e-06
  mlflow_run_id: 
  mlflow_tracking_uri: 
  model_name_or_path: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
  name: example-name-5x0q0
  results_directory: 
  train_val_split_ratio: 1.0
  warmup_steps: 10
  weight_decay: 0.1


### Launch

`wait=False` returns immediately. The job takes **25-30 minutes** for these 8 epochs.
A sizeable and fixed part of that is SageMaker provisioning the compute rather than
training, so the wall-clock does not scale with epochs the way you might expect:
halving the epochs does not halve the time.

In [9]:
training_job = trainer.train(wait=False)
TRAINING_JOB_NAME = training_job.training_job_name
print(f"launched: {TRAINING_JOB_NAME}")

[09/10/26 02:00:26] INFO     Training Job Name: contractnli-sft-20260910020026                   ]8;id=413491;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/sft_trainer.py\sft_trainer.py]8;;\:]8;id=200228;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/sft_trainer.py#348\348]8;;\

                    INFO     Found 1 MLflow apps: [('mflowapp', 'Updated', '3.10.1')]         ]8;id=430354;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=333355;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py#219\219]8;;\

                    INFO     Resolved MLflow app:                                             ]8;id=265100;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=785842;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py#242\242]8;;\
                             arn:aws:sagemaker:us-east-1:492681118881:mlflow-app/app-LU5W7ACY                      
                             LH65 (status: Updated, version: 3.10.1)                                               

                    INFO     MLflow resource ARN:                                            ]8;id=276998;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py\finetune_utils.py]8;;\:]8;id=117575;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/common_utils/finetune_utils.py#1080\1080]8;;\
                             arn:aws:sagemaker:us-east-1:492681118881:mlflow-app/app-LU5W7AC                       
                             YLH65                                                                                 

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[09/10/26 02:00:27] INFO     Creating training_job resource.                                     ]8;id=932487;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=224596;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31238\31238]8;;\

launched: contractnli-sft-20260910020026


In [10]:
from sagemaker.core.resources import TrainingJob

response = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
print(f"Status: {response.training_job_status}")
print(f"Secondary: {response.secondary_status}")

Status: InProgress
Secondary: Starting


Poll until the job reaches `Completed`, then continue to **notebook 3** to find
out whether any of this actually helped.

In [11]:
import time

from sagemaker.core.resources import TrainingJob

while True:
    j = TrainingJob.get(training_job_name=TRAINING_JOB_NAME)
    print(f"{j.training_job_status} / {j.secondary_status}")
    if j.training_job_status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(120)

InProgress / Starting
InProgress / Pending
InProgress / Pending
InProgress / Downloading
InProgress / Downloading
InProgress / Downloading
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Training
InProgress / Uploading
InProgress / Uploading
InProgress / Uploading
Completed / Completed
